In [ ]:
import numpy as np 
import torch 

In [ ]:
# implement attention
# for simplicity, we will avoid using masks. 

class Attention:
    def __init__(self, d_embed, num_heads):
        self.d_embed = d_embed
        self.num_heads = num_heads
        self.head_dim = d_embed // num_heads

        self.q_proj = np.random.randn(d_embed, d_embed)
        self.k_proj = np.random.randn(d_embed, d_embed)
        self.v_proj = np.random.randn(d_embed, d_embed)

    def forward(self, x):
        # X = [B,T,C]
        B, T, C = x.shape

        # project the query, key, value with numpy
        q = x @ self.q_proj # [B,T,C]
        k = x @ self.k_proj # [B,T,C]
        v = x @ self.v_proj # [B,T,C]

        # split to [B,T,nH,D] and transpose to [B,nh,T,D]
        q = q.reshape(B, T, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        k = k.reshape(B, T, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        v = v.reshape(B, T, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)

        # compute the attention weights
        logits = q @ k.transpose(-2, -1) / np.sqrt(self.head_dim) # [B,nh,T,T]
        # weights = np.softmax(weights, axis=-1)

        # can we do softmax without numpy?
        max_val = np.max(logits, axis=-1, keepdims=True)
        logits = np.exp(logits - max_val)
        logits = logits / np.sum(logits, axis=-1, keepdims=True)
        
        # compute the attention output
        out = logits @ v # [B,nh,T,D]
        out = out.transpose(0, 2, 1, 3).reshape(B, T, C)

        return out

In [ ]:
# implement FalshAttention, the key is to use tiling. 
# You will not be asked this algorihtm in the the interview. 

# this is too comlicated for interview. 

class FlashAttention:
    def __init__(self, d_embed, num_heads):
        self.d_embed = d_embed
        self.num_heads = num_heads
        self.head_dim = d_embed // num_heads
        self.scale = 1.0 / np.sqrt(self.head_dim)

    def forward(self, q, k, v):
        """
        Expects q, k, v already projected and reshaped to [B, nh, T, d]
        """
        B, nh, T, d = q.shape
        # Resulting output matrix
        O = np.zeros_like(q)
        
        # Tiling parameters
        Br = 64 # Row block size
        Bc = 64 # Column block size
        
        tr = (T + Br - 1) // Br
        tc = (T + Bc - 1) // Bc

        for b in range(B):
            for h in range(nh):
                # Process blocks of rows (Queries)
                for i in range(tr):
                    start_r, end_r = i * Br, min((i + 1) * Br, T)
                    
                    # Load Qi into SRAM (simulated)
                    Qi = q[b, h, start_r:end_r, :] # [Br, d]
                    
                    # Initialize row-state registers for this block of rows
                    # m and d are vectors of size [Br]
                    m_i = np.full((end_r - start_r, 1), -np.inf)
                    d_i = np.zeros((end_r - start_r, 1))
                    O_i = np.zeros((end_r - start_r, d)) # Accumulator [Br, d]

                    # Process blocks of columns (Keys/Values)
                    for j in range(tc):
                        start_c, end_c = j * Bc, min((j + 1) * Bc, T)
                        
                        Kj = k[b, h, start_c:end_c, :] # [Bc, d]
                        Vj = v[b, h, start_c:end_c, :] # [Bc, d]

                        # 1. Compute Local Scores (The segment)
                        S_ij = (Qi @ Kj.T) * self.scale # [Br, Bc]

                        # 2. Compute Local Statistics
                        m_local = np.max(S_ij, axis=1, keepdims=True) # [Br, 1]
                        P_ij = np.exp(S_ij - m_local) # [Br, Bc]
                        d_local = np.sum(P_ij, axis=1, keepdims=True) # [Br, 1]

                        # 3. Compute Rescale Factors
                        m_new = np.maximum(m_i, m_local) # [Br, 1]
                        alpha = np.exp(m_i - m_new)      # [Br, 1]
                        beta = np.exp(m_local - m_new)   # [Br, 1]

                        # 4. Update Running Denominator
                        d_new = (d_i * alpha) + (d_local * beta) # [Br, 1]

                        # 5. Update Running Output (Fused step)
                        # We rescale the old O_i and add the new weighted V_j contribution
                        O_i = (O_i * alpha) + (beta * (P_ij @ Vj))

                        # 6. Commit new state to registers
                        m_i = m_new
                        d_i = d_new

                    # 7. Final Normalization for the row block
                    O[b, h, start_r:end_r, :] = O_i / d_i

        return O

In [3]:
# implement paged attention (if possible)

In [ ]:
# Grouped Query Attention
# key and values are calculated once and reused for all heads
# query is calculated for each head

class GroupedQueryAttention:
    def __init__(self, d_embed, num_heads):
        self.d_embed = d_embed
        